# Current-Run Motion-Regime Characterization

This notebook characterizes the current canonical OASIS and 4DCT displacement fields for the MICCAI off-grid workshop paper. The fields are method-estimated dense DVFs, not ground-truth physical motion.

Primary methods:

- OASIS seed-3 shuffled first10: `SINR cps4 alpha=1000`.
- 4DCT/LungCT cases 01-10: `IDIR original/default`.

Sensitivity methods:

- OASIS seed-3 shuffled first10: `DirectCP cps4 alpha=1000`.
- 4DCT/LungCT cases 01-10: `DirectCP 32-16-8-4 pTV LCC + isoTV`.

Foreground convention is part of the scientific claim. OASIS uses fixed raw image-positive foreground (`fixed_image > 0`), not `seg > 0`. 4DCT uses the fixed T00 lung mask on the cropped 1mm pTV-prep grid.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts/reports/motion_regime_current_methods.py").exists():
    candidate = Path("/homebase/code_sync/inrorspline")
    if (candidate / "scripts/reports/motion_regime_current_methods.py").exists():
        REPO_ROOT = candidate

OUT_DIR = REPO_ROOT / "experiments/2026-06-25_motion_regime_seed3_first10_current_methods"
SCRIPT = REPO_ROOT / "scripts/reports/motion_regime_current_methods.py"

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

print(f"Repo: {REPO_ROOT}")
print(f"Output: {OUT_DIR}")
print(f"Script: {SCRIPT}")

## Execute Analysis

This cell runs the reusable analysis script and keeps the generated outputs under the experiment root. It intentionally fails fast if any selected input is missing, if the OASIS mask audit is not `fixed raw image > 0`, if the 4DCT fixed T00 lung-mask convention is not present, or if the expected 10 fields per dataset/method are unavailable.

In [ ]:
cmd = [sys.executable, str(SCRIPT), "--out-dir", str(OUT_DIR), "--run-sanity-tests"]
print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_ROOT, check=True)

In [ ]:
def require(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return path

def read_csv(name):
    return pd.read_csv(require(OUT_DIR / name))

def show_png(name, width=1100):
    display(Image(filename=str(require(OUT_DIR / name)), width=width))

def show_md(name):
    display(Markdown(require(OUT_DIR / name).read_text()))

## Data Audit

The audit table is the first check to read before interpreting any figure. It confirms the selected method, case identifier, field shape, component order, mask rule, and foreground voxel count. A mismatch here means the quantitative results should not be used in the paper.

In [ ]:
audit = read_csv("data_audit.csv")
display(audit)

counts = audit.groupby(["dataset", "role", "method_key"], as_index=False).size()
display(counts)

## Tables For Paper Use

The organized table is the compact paper-facing version. It is one row per dataset/method and contains the columns most directly tied to deformation-regime fit: amplitude, thresholded large motion, local regularity, Jacobian behavior, correlation length, and coarse-grid explained energy.

The contrast table reports 4DCT/OASIS ratios for the same columns. Values above 1 indicate that 4DCT is larger than OASIS on that property; values below 1 indicate the opposite. Ratios should be interpreted as regime contrasts, not method superiority, because the primary rows compare different dataset protocols and different current methods.

In [ ]:
paper = read_csv("tables/paper_table_organized.csv")
contrast = read_csv("tables/contrast_table_4dct_over_oasis.csv")
display(paper)
display(contrast)

The comprehensive per-case table is intentionally wider. Use it for debugging, supplementary material, or selecting representative cases. The paper table should usually be preferred for the main manuscript unless a per-case result is being discussed explicitly.

In [ ]:
case_metrics = read_csv("tables/case_metrics_comprehensive.csv")
display(case_metrics)

aggregate = read_csv("tables/aggregate_metrics_comprehensive.csv")
display(aggregate)

## Figure 1: Amplitude Distributions

These boxplots summarize foreground displacement magnitude. Mean magnitude captures typical deformation size. P95 and P99 capture tail behavior without being dominated by single voxels. `% voxels > 10mm` is the practical large-motion fraction. For the paper argument, this figure answers whether the current 4DCT fields occupy a larger-amplitude deformation regime than the current OASIS fields.

In [ ]:
show_png("plots/primary_amplitude_boxplots.png", width=1200)

Interpretation rule: amplitude alone is not enough to justify the off-grid/local-control story. A large displacement field that is spatially coherent can still be explainable by a coarse model, while a smaller field can be difficult if it contains high-frequency local variation. Read Figure 1 together with Figures 2-5.

## Figure 2: Smoothness, Regularity, And Jacobian Properties

Diffusion energy measures first-derivative magnitude. Bending energy measures second-derivative burden. Displacement TV and strain summarize local variation. `logJ std` reports volumetric variability. `J<=0 %` reports folding in the estimated transformation. These metrics test whether a method-dataset pair is locally regular or whether it requires sharp, high-curvature corrections.

In [ ]:
show_png("plots/primary_smoothness_boxplots.png", width=1200)

Interpretation rule: lower smoothness penalties are not automatically better across datasets. In this notebook they characterize the estimated deformation regime. If 4DCT has larger amplitude but comparable or lower local irregularity, that supports the idea that respiratory motion is large but coherent. If 4DCT also has much larger bending or folding, the argument should shift toward local complexity and optimization burden.

## Figure 3: Spatial Coherence

Vector autocorrelation is computed at lags 2, 4, 8, 16, 32, and 64 mm inside the correct foreground mask. Slow decay means nearby and moderately distant vectors point in similar directions and magnitudes. The semivariogram is complementary: higher short-lag values mean stronger local variation.

In [ ]:
show_png("plots/primary_autocorrelation.png", width=900)
show_png("plots/primary_semivariogram.png", width=900)

Interpretation rule: a longer correlation length suggests that coarse/local-control parameterizations can cover much of the field without needing many independent local degrees of freedom. A steep autocorrelation drop or high short-lag semivariogram indicates high local variability.

## Figure 4: Coarse-Grid Explainability

Coarse-grid explainability downsamples each dense field to control grids at 64, 32, 16, 8, 4, and 2 mm, upsamples back, and measures retained displacement energy on the foreground. Residual P95 reports the remaining displacement magnitude not explained by that spacing. This is the most direct analysis of whether a local-control grid can represent the estimated field.

In [ ]:
show_png("plots/primary_coarse_explained_energy.png", width=900)
show_png("plots/primary_coarse_residual_p95.png", width=900)

Interpretation rule: high explained energy at 16/32/64 mm means the field is mostly low-dimensional at that physical scale. A large residual P95 at coarse spacing means the method is using finer structure that a coarse grid cannot capture. These columns are included in the organized paper table because they map directly to spline/local-control fit.

## Figure 5: Low-Pass Explainability

Gaussian low-pass explainability asks how much displacement energy remains after smoothing each component at sigma 2, 4, 8, 16, and 32 mm. It is a complementary frequency-domain-style view of the same regime question as Figure 4, but without imposing a specific grid parameterization.

In [ ]:
show_png("plots/primary_lowpass_explained_energy.png", width=900)

Interpretation rule: high low-pass explained energy at large sigma indicates broad, smooth motion. A steep drop at large sigma means important deformation energy exists at smaller spatial scales. Compare this with coarse-grid explainability to separate general smoothness from grid-representation fit.

## Figure 6: Representative Displacement-Magnitude Slices

These slices show median-P95 and high-P95 examples for each primary method. The shared color scale is important: it makes cross-dataset amplitude differences visually comparable. The slices are for qualitative sanity checking and figure selection, not for claiming anatomical ground truth.

In [ ]:
show_png("plots/primary_representative_magnitude_slices.png", width=1300)

## Sensitivity Analysis

The sensitivity section repeats the key aggregate views with OASIS DirectCP cps4 and 4DCT DirectCP multistage. These rows are not co-primary. They answer whether the observed deformation-regime contrast is robust to replacing the primary method with the backup/current alternative on each dataset.

In [ ]:
show_png("plots/all_methods_amplitude_boxplots.png", width=1200)
show_png("plots/all_methods_smoothness_boxplots.png", width=1200)
show_png("plots/all_methods_autocorrelation.png", width=900)
show_png("plots/all_methods_coarse_explained_energy.png", width=900)

Interpretation rule: if primary and sensitivity contrasts point in the same direction, the paper can frame the result as dataset/regime-driven rather than a single-method artifact. If the sensitivity rows disagree, the report should explicitly state that the current conclusion depends on method family or regularization.

## Generated Markdown Report

The script writes a compact Markdown report from the same tables and figures. Use this as a manuscript-facing summary or as the source for figure captions.

In [ ]:
show_md("review_report.md")